# 🎬 ĐẾM NGƯỜI & PHƯƠNG TIỆN — YOLOv8 + supervision (Colab & Kaggle)

Hệ thống đếm **chỉ dùng YOLOv8 + supervision** cho **2 bài toán YOLO làm tốt**:
🚗 **phương tiện** · 🚶 **người**. Mỗi bài có sẵn kịch bản **cắt VẠCH** (vào/ra) và
**đếm VÙNG** (occupancy). YOLO đã chỉnh **ít bỏ sót** (yolov8x, imgsz1280, conf0.15).

> Cần **GPU** (Colab: Runtime→T4 · Kaggle: Settings→Accelerator→GPU). YOLO nhanh nên
> chạy cả bộ chỉ vài phút.

## 1) Cài đặt + tải code

In [ ]:
import os
if os.path.isdir('/kaggle/working'):      WORK = '/kaggle/working'
elif os.path.isdir('/content'):           WORK = '/content'
else:                                      WORK = os.path.abspath('.')
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
print('📂 WORK =', WORK)
REPO = 'https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git'; BRANCH = 'claude/locate-anything-test-suite-xwju2f'
if not os.path.isdir(f'{WORK}/VisionOS/.git'):
    os.system(f'git clone -q {REPO} {WORK}/VisionOS')
os.chdir(f'{WORK}/VisionOS')
os.system(f'git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset --hard -q origin/{BRANCH}')
os.chdir(f'{WORK}/VisionOS/VisionOS')
print('📁 cwd =', os.getcwd())
os.system("pip install -q ultralytics 'supervision>=0.21' opencv-python-headless")
import torch
print('🖥️  GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else '❌ CHƯA BẬT GPU! Colab: Runtime→T4 · Kaggle: Settings→Accelerator→GPU')

## 2) ✅ Kiểm tra video tải được (nhanh, không cần model)

In [ ]:
!python run_scenarios.py --download-only

## 3) ⭐ XEM TRƯỚC vạch/vùng trên frame thật
Vàng = **vạch** cắt (vào/ra), xanh = **vùng** đếm, lưới = %. Muốn đổi → báo toạ độ %
hoặc dùng `--line 'x1,y1,x2,y2'` / `--zone 'x1,y1;x2,y2;…'`.

In [ ]:
import os, glob
from IPython.display import Image, display, Markdown
WORK = globals().get('WORK') or ('/kaggle/working' if os.path.isdir('/kaggle/working') else '/content')
!python run_scenarios.py --preview {WORK}/prev
allp = f'{WORK}/prev/_ALL.jpg'
if os.path.exists(allp):
    display(Markdown('### 🧩 TỔNG HỢP tất cả kịch bản')); display(Image(filename=allp, width=940))

## 4) ⭐ ĐẾM HẾT — người + phương tiện (YOLO, lưu video)
Chạy MỌI kịch bản của 2 bài (gồm bài **cắt VẠCH** và bài **đếm VÙNG**), `stride=1`
(đếm chuẩn, không bỏ frame), lưu video annotate (màu theo LỚP: car/truck/bus mỗi loại
một màu) vào `scen_out/`.

In [ ]:
WORK = globals().get('WORK') or '/content'
# 🚗 PHƯƠNG TIỆN (car/truck/bus/moto) — vạch vào/ra + đếm trong vùng
!python run_scenarios.py --task vehicles --max-frames 300 --save-dir {WORK}/scen_out

In [ ]:
WORK = globals().get('WORK') or '/content'
# 🚶 NGƯỜI — vạch vào/ra + đếm trong vùng (nhiều cảnh: đi bộ, ga tàu, siêu thị, quảng trường)
!python run_scenarios.py --task people --max-frames 300 --save-dir {WORK}/scen_out

## 5) 🎥 Xem / tải video output

In [ ]:
import glob, os
from IPython.display import Video, display, Markdown
WORK = globals().get('WORK') or ('/kaggle/working' if os.path.isdir('/kaggle/working') else '/content')
vids = sorted(glob.glob(f'{WORK}/scen_out/**/*.mp4', recursive=True))
print(f'{len(vids)} video output:')
for p in vids: print('  ', p)
for src in vids[:4]:
    dst = src.replace('.mp4', '_h264.mp4')
    os.system(f'ffmpeg -y -loglevel error -i "{src}" -vcodec libx264 -pix_fmt yuv420p "{dst}"')
    display(Markdown(f'**{os.path.basename(src)}**')); display(Video(dst, embed=True, width=680))

### 📖 Đọc số đếm
- **IN/OUT/total** = số vật ĐI QUA VẠCH (đếm vào/ra 1 lối).
- **trong_vùng / đỉnh_vùng** = số vật ĐANG trong vùng / lúc đông nhất (occupancy).
- **tracks** = tổng số vật KHÁC NHAU thấy (≈ "có bao nhiêu người/xe").
- **det/frame** = TB vật/khung (đo sức detect của YOLO).

**Bỏ sót nhiều?** Tăng recall: đặt env `YOLO_AUGMENT=1` (TTA) hoặc `YOLO_IMGSZ=1536`.
Muốn nhanh hơn: `YOLO_WEIGHTS=yolov8m.pt`. **Đếm VẠCH=0?** vạch lệch dòng đi — đổi bằng
`--line 'x1,y1,x2,y2'` (%) rồi chạy `--only <tên video>`.